# Notebook 1: Data Loading, Visualization & Diagnostics

This notebook walks through the full data pipeline:
1. Loading raw experimental trajectories from CSV
2. PCA fitting and variance analysis
3. Plotting trajectories in PC space
4. Fragment sampler: visual inspection of multi-segment sampling
5. Class-balanced sampling weights

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import Counter

# Ensure the repo root is on the path
REPO_ROOT = "/net/trapnell/vol1/home/nlammers/projects/repositories/morphseq"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

---
## 1. Load experimental trajectories

Point `build_dir` at the build06 output folder. Pick a few experiments to start (loading all 43 takes a while).

In [ ]:
BUILD_DIR = "/net/trapnell/vol1/home/mdcolon/proj/morphseq/morphseq_playground/metadata/build06_output"

# Pick a small set of experiments to inspect.
# Set to None to auto-discover all experiments in BUILD_DIR.
EXPERIMENT_IDS = ["20251207_pbx", "20251121"]

N_COMPONENTS = 10   # PCA components to keep
SCALE = True        # Standardize z_mu_b before PCA

In [ ]:
from dev.dynamo.data.loading import load_trajectories

dataset = load_trajectories(
    experiment_ids=EXPERIMENT_IDS,
    build_dir=BUILD_DIR,
    n_components=N_COMPONENTS,
    scale=SCALE,
    verbose=True,
)

print(f"\n--- Summary ---")
print(f"  Trajectories:  {len(dataset)}")
print(f"  PC dimensions: {dataset.n_components}")
print(f"  Classes:       {dataset.class_names}")
print(f"  Class→idx:     {dataset.class_to_idx}")

### Inspect a single trajectory

In [ ]:
traj = dataset.trajectories[0]
print(f"Embryo:            {traj.embryo_id}")
print(f"Experiment:        {traj.experiment_id}")
print(f"Class:             {traj.perturbation_class}")
print(f"Temperature:       {traj.temperature}°C")
print(f"Frames:            {len(traj.trajectory)}")
print(f"Trajectory shape:  {traj.trajectory.shape}")
print(f"Time range:        {traj.time_seconds[0]:.0f}s – {traj.time_seconds[-1]:.0f}s")
print(f"Experiment Δt:     {traj.delta_t:.1f}s")

# Check inter-frame intervals
diffs = np.diff(traj.time_seconds)
print(f"\nInter-frame Δt:  min={diffs.min():.1f}s  median={np.median(diffs):.1f}s  max={diffs.max():.1f}s")

### Trajectory length distribution

In [ ]:
lengths = [len(t.trajectory) for t in dataset.trajectories]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of trajectory lengths
axes[0].hist(lengths, bins=30, edgecolor="k", alpha=0.7)
axes[0].axvline(np.median(lengths), color="red", ls="--", label=f"median={int(np.median(lengths))}")
axes[0].set_xlabel("Trajectory length (frames)")
axes[0].set_ylabel("Count")
axes[0].set_title("Trajectory length distribution")
axes[0].legend()

# Distribution of inter-frame Δt across all trajectories
all_dts = np.concatenate([np.diff(t.time_seconds) for t in dataset.trajectories])
axes[1].hist(all_dts, bins=50, edgecolor="k", alpha=0.7)
axes[1].axvline(np.median(all_dts), color="red", ls="--", label=f"median={np.median(all_dts):.0f}s")
axes[1].set_xlabel("Inter-frame Δt (seconds)")
axes[1].set_ylabel("Count")
axes[1].set_title("Inter-frame interval distribution (after repair)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2. PCA analysis

Examine variance explained and the structure of the PC embedding.

In [ ]:
pca = dataset.pca
var_ratio = pca.explained_variance_ratio_
cum_var = np.cumsum(var_ratio)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(var_ratio) + 1), var_ratio, color="steelblue", edgecolor="k")
axes[0].set_xlabel("PC")
axes[0].set_ylabel("Variance explained")
axes[0].set_title("Per-component variance")
axes[0].set_xticks(range(1, len(var_ratio) + 1))

axes[1].plot(range(1, len(cum_var) + 1), cum_var, "o-", color="steelblue")
axes[1].axhline(0.9, color="gray", ls="--", alpha=0.5, label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title("Cumulative variance")
axes[1].set_xticks(range(1, len(cum_var) + 1))
axes[1].legend()

plt.tight_layout()
plt.show()

for i, (v, c) in enumerate(zip(var_ratio, cum_var)):
    print(f"  PC{i+1:2d}: {v*100:5.1f}%  (cumulative: {c*100:5.1f}%)")

---
## 3. Plot trajectories in PC space

Color by perturbation class to see how different genotypes separate.

In [ ]:
# Build a color map for classes
class_names = dataset.class_names
n_classes = len(class_names)
cmap = plt.cm.get_cmap("tab10", max(n_classes, 3))
class_colors = {cls: cmap(i) for i, cls in enumerate(class_names)}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pc_pairs = [(0, 1), (0, 2), (1, 2)]  # PC1 vs PC2, PC1 vs PC3, PC2 vs PC3
max_trajs = 80  # subsample for readability

rng = np.random.default_rng(42)
show_idx = rng.choice(len(dataset.trajectories), size=min(max_trajs, len(dataset)), replace=False)

for ax, (pc_x, pc_y) in zip(axes, pc_pairs):
    for cls in class_names:
        ax.plot([], [], color=class_colors[cls], label=cls, lw=2)
    
    for i in show_idx:
        t = dataset.trajectories[i]
        color = class_colors[t.perturbation_class]
        ax.plot(t.trajectory[:, pc_x], t.trajectory[:, pc_y],
                color=color, alpha=0.4, lw=0.8)
        # Start marker
        ax.scatter(t.trajectory[0, pc_x], t.trajectory[0, pc_y],
                   color=color, s=15, zorder=5, marker="o", edgecolors="k", linewidths=0.3)
    
    ax.set_xlabel(f"PC{pc_x+1}")
    ax.set_ylabel(f"PC{pc_y+1}")
    ax.set_title(f"PC{pc_x+1} vs PC{pc_y+1}")
    ax.legend(fontsize=7, loc="best")

plt.suptitle("Trajectories in PC space (colored by perturbation class)", fontsize=13)
plt.tight_layout()
plt.show()

### Time-colored trajectories (developmental progression)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show a handful of trajectories colored by time
n_show = min(12, len(dataset))
show_idx = rng.choice(len(dataset.trajectories), size=n_show, replace=False)

for ax_idx, (pc_x, pc_y) in enumerate([(0, 1), (0, 2)]):
    ax = axes[ax_idx]
    for i in show_idx:
        t = dataset.trajectories[i]
        # Normalize time to [0, 1] for color mapping
        t_norm = (t.time_seconds - t.time_seconds[0]) / max(t.time_seconds[-1] - t.time_seconds[0], 1)
        
        for j in range(len(t.trajectory) - 1):
            ax.plot(t.trajectory[j:j+2, pc_x], t.trajectory[j:j+2, pc_y],
                    color=plt.cm.viridis(t_norm[j]), lw=1.2, alpha=0.7)
        ax.scatter(t.trajectory[0, pc_x], t.trajectory[0, pc_y],
                   color="green", s=30, zorder=5, marker="o", edgecolors="k", linewidths=0.5)
        ax.scatter(t.trajectory[-1, pc_x], t.trajectory[-1, pc_y],
                   color="red", s=30, zorder=5, marker="s", edgecolors="k", linewidths=0.5)
    
    ax.set_xlabel(f"PC{pc_x+1}")
    ax.set_ylabel(f"PC{pc_y+1}")
    ax.set_title(f"PC{pc_x+1} vs PC{pc_y+1}")

# Add colorbar
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=axes, shrink=0.8)
cbar.set_label("Normalized time (0=start, 1=end)")

plt.suptitle("Trajectories colored by developmental time (green=start, red=end)", fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Fragment sampler: visual inspection

The `FragmentDataset` draws random fragments from trajectories for training. Each sample consists of:
- A **context** window (contiguous sub-trajectory)
- **M target** frames at random horizons ahead of the context
- The **predecessor** of each target (for teacher-forced training)

We want to verify:
- Fragments are contiguous sub-sequences of the original trajectory
- Targets land at correct positions relative to context end
- Time deltas are consistent with the original trajectory's timestamps

In [ ]:
from dev.dynamo.data.dataset import FragmentDataset, fragment_collate_fn

frag_ds = FragmentDataset(
    dataset,
    min_context=3,
    max_context=None,
    horizons=(1, 2, 3, 4),
    epoch_length=500,
    gamma=0.5,
    n_targets=3,  # M=3 targets per fragment for visualization
)

print(f"FragmentDataset: {len(frag_ds)} samples/epoch")
print(f"  Valid trajectories: {len(frag_ds.valid_indices)} / {len(dataset)}")
print(f"  Horizons: {frag_ds.horizons}")
print(f"  n_targets (M): {frag_ds.n_targets}")

### Visualize sampled fragments overlaid on their parent trajectories

For each sample, we show the full parent trajectory (gray), the context window (blue), the targets (red), and the predecessors (orange).

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()

for panel_idx in range(12):
    ax = axes[panel_idx]
    sample = frag_ds[panel_idx]  # stochastic — each call draws a new sample
    
    # Recover parent trajectory
    traj_idx = sample["embryo_idx"].item()
    traj = dataset.trajectories[traj_idx]
    
    context = sample["context"].numpy()        # (L, D)
    targets = sample["targets"].numpy()         # (M, D)
    predecessors = sample["predecessors"].numpy()  # (M, D)
    
    pc_x, pc_y = 0, 1
    
    # Full trajectory (gray)
    ax.plot(traj.trajectory[:, pc_x], traj.trajectory[:, pc_y],
            color="lightgray", lw=1, zorder=1, label="full trajectory")
    
    # Context window (blue)
    ax.plot(context[:, pc_x], context[:, pc_y],
            color="steelblue", lw=2.5, zorder=3, label=f"context (L={len(context)})")
    ax.scatter(context[0, pc_x], context[0, pc_y],
               color="steelblue", s=60, zorder=4, marker="o", edgecolors="k", linewidths=0.5)
    ax.scatter(context[-1, pc_x], context[-1, pc_y],
               color="steelblue", s=60, zorder=4, marker=">", edgecolors="k", linewidths=0.5)
    
    # Predecessors (orange diamonds)
    ax.scatter(predecessors[:, pc_x], predecessors[:, pc_y],
               color="orange", s=80, zorder=5, marker="D", edgecolors="k", linewidths=0.5,
               label=f"predecessors (M={len(predecessors)})")
    
    # Targets (red stars)
    ax.scatter(targets[:, pc_x], targets[:, pc_y],
               color="red", s=100, zorder=5, marker="*", edgecolors="k", linewidths=0.3,
               label=f"targets (M={len(targets)})")
    
    # Draw arrows from predecessor to target
    for m in range(len(targets)):
        ax.annotate("", xy=(targets[m, pc_x], targets[m, pc_y]),
                    xytext=(predecessors[m, pc_x], predecessors[m, pc_y]),
                    arrowprops=dict(arrowstyle="->", color="red", lw=1.5, alpha=0.6))
    
    ax.set_title(f"embryo={traj.embryo_id[:12]}\nclass={traj.perturbation_class}", fontsize=8)
    ax.set_xlabel(f"PC{pc_x+1}", fontsize=8)
    ax.set_ylabel(f"PC{pc_y+1}", fontsize=8)
    ax.tick_params(labelsize=7)
    if panel_idx == 0:
        ax.legend(fontsize=6, loc="best")

plt.suptitle("Fragment sampling: context (blue) + predecessors (orange) + targets (red)\n"
             "on parent trajectory (gray)", fontsize=13)
plt.tight_layout()
plt.show()

### Sanity check: verify fragments are contiguous sub-sequences

For each sample, we find where the context window matches in the parent trajectory and check that the targets also fall at the right positions.

In [ ]:
n_checks = 200
n_issues = 0

for i in range(n_checks):
    sample = frag_ds[i]
    traj_idx = sample["embryo_idx"].item()
    traj = dataset.trajectories[traj_idx]
    
    context = sample["context"].numpy()
    targets = sample["targets"].numpy()
    predecessors = sample["predecessors"].numpy()
    
    # Find context start in parent trajectory
    # Context[0] should exactly match some frame in the trajectory
    dists = np.linalg.norm(traj.trajectory - context[0], axis=1)
    start_idx = np.argmin(dists)
    
    if dists[start_idx] > 1e-6:
        print(f"  Sample {i}: context start not found in trajectory! min_dist={dists[start_idx]:.6f}")
        n_issues += 1
        continue
    
    # Check that context is contiguous
    L = len(context)
    expected_context = traj.trajectory[start_idx:start_idx + L]
    if not np.allclose(context, expected_context, atol=1e-6):
        print(f"  Sample {i}: context is NOT a contiguous sub-sequence!")
        n_issues += 1
        continue
    
    # Check that each target exists in the trajectory
    context_end = start_idx + L  # exclusive
    for m in range(len(targets)):
        # Target should be at some index >= context_end
        t_dists = np.linalg.norm(traj.trajectory[context_end-1:] - targets[m], axis=1)
        t_idx = np.argmin(t_dists)
        if t_dists[t_idx] > 1e-6:
            print(f"  Sample {i}, target {m}: not found in trajectory! min_dist={t_dists[t_idx]:.6f}")
            n_issues += 1
        
        # Predecessor should be exactly one frame before target in trajectory
        p_dists = np.linalg.norm(traj.trajectory - predecessors[m], axis=1)
        p_idx = np.argmin(p_dists)
        target_abs_idx = context_end - 1 + t_idx
        if p_idx != target_abs_idx - 1:
            # Check if predecessor is truly the frame before target
            if p_dists[p_idx] > 1e-6:
                print(f"  Sample {i}, target {m}: predecessor mismatch")
                n_issues += 1

    # Check time_deltas match trajectory timestamps
    td = sample["time_deltas"].numpy()
    expected_td = np.diff(traj.time_seconds[start_idx:start_idx + L])
    if not np.allclose(td, expected_td, atol=1e-4):
        print(f"  Sample {i}: time_deltas mismatch!")
        n_issues += 1

if n_issues == 0:
    print(f"All {n_checks} samples passed sanity checks.")
else:
    print(f"\n{n_issues} issues found in {n_checks} samples.")

### Visualize multiple fragments drawn from the same trajectory

This shows how different random context windows tile a single trajectory.

In [ ]:
# Pick a long trajectory to show multiple fragments from
long_trajs = [(i, t) for i, t in enumerate(dataset.trajectories) if len(t.trajectory) >= 15]
if not long_trajs:
    long_trajs = [(0, dataset.trajectories[0])]
example_idx, example_traj = long_trajs[0]

# Create a dedicated FragmentDataset with just this trajectory to force sampling from it
from dev.dynamo.data.loading import TrajectoryDataset
import dataclasses

single_ds = dataclasses.replace(dataset, trajectories=[example_traj])
single_frag = FragmentDataset(single_ds, min_context=3, horizons=(1, 2, 3, 4),
                              epoch_length=100, gamma=0.0, n_targets=2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
n_fragments = 8
frag_colors = plt.cm.Set2(np.linspace(0, 1, n_fragments))

for ax_idx, (pc_x, pc_y) in enumerate([(0, 1), (0, 2)]):
    ax = axes[ax_idx]
    
    # Plot the full trajectory as a thick gray line with frame markers
    ax.plot(example_traj.trajectory[:, pc_x], example_traj.trajectory[:, pc_y],
            color="lightgray", lw=3, zorder=1)
    ax.scatter(example_traj.trajectory[:, pc_x], example_traj.trajectory[:, pc_y],
               color="lightgray", s=25, zorder=2, edgecolors="gray", linewidths=0.3)
    
    # Draw fragments
    for f_idx in range(n_fragments):
        sample = single_frag[f_idx]
        ctx = sample["context"].numpy()
        tgts = sample["targets"].numpy()
        color = frag_colors[f_idx]
        
        ax.plot(ctx[:, pc_x], ctx[:, pc_y], color=color, lw=2, alpha=0.8,
                label=f"frag {f_idx+1} (L={len(ctx)})" if ax_idx == 0 else "")
        ax.scatter(tgts[:, pc_x], tgts[:, pc_y], color=color, s=80,
                   marker="*", edgecolors="k", linewidths=0.3, zorder=6)
    
    ax.set_xlabel(f"PC{pc_x+1}")
    ax.set_ylabel(f"PC{pc_y+1}")
    ax.set_title(f"PC{pc_x+1} vs PC{pc_y+1}")

axes[0].legend(fontsize=7, loc="best")
plt.suptitle(f"Multiple sampled fragments from one trajectory (embryo={example_traj.embryo_id[:15]})\n"
             f"Gray=full trajectory, colored lines=context, stars=targets", fontsize=12)
plt.tight_layout()
plt.show()

### Fragment timing diagnostics

Verify that `horizon_dts` and `context_to_target_dts` are physically reasonable.

In [ ]:
# Collect timing stats from many samples
n_samples = 500
all_horizon_dts = []
all_ctx_to_tgt_dts = []
all_ctx_time_deltas = []
all_ctx_lengths = []

for i in range(n_samples):
    s = frag_ds[i]
    all_horizon_dts.extend(s["horizon_dts"].numpy().tolist())
    all_ctx_to_tgt_dts.extend(s["context_to_target_dts"].numpy().tolist())
    all_ctx_time_deltas.extend(s["time_deltas"].numpy().tolist())
    all_ctx_lengths.append(s["context"].shape[0])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(all_horizon_dts, bins=40, edgecolor="k", alpha=0.7, color="salmon")
axes[0, 0].set_xlabel("horizon_dt (predecessor→target, seconds)")
axes[0, 0].set_title("Teacher-forced step size")

axes[0, 1].hist(all_ctx_to_tgt_dts, bins=40, edgecolor="k", alpha=0.7, color="mediumpurple")
axes[0, 1].set_xlabel("context_to_target_dt (last context→target, seconds)")
axes[0, 1].set_title("Context-to-target time gap")

axes[1, 0].hist(all_ctx_time_deltas, bins=40, edgecolor="k", alpha=0.7, color="seagreen")
axes[1, 0].set_xlabel("Within-context Δt (seconds)")
axes[1, 0].set_title("Inter-frame Δt within context")

axes[1, 1].hist(all_ctx_lengths, bins=range(1, max(all_ctx_lengths) + 2),
                edgecolor="k", alpha=0.7, color="steelblue", align="left")
axes[1, 1].set_xlabel("Context length (frames)")
axes[1, 1].set_title("Context window sizes")

plt.suptitle("Fragment timing diagnostics", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Class-balanced sampling

The sampler reweights trajectory selection so rare classes are seen more often.
Parameter γ controls the strength:
- γ=0: natural frequencies (no rebalancing)
- γ=0.5: square-root weighting (default)
- γ=1: uniform across classes

In [ ]:
# Show natural class distribution
class_counts = Counter(t.perturbation_class for t in dataset.trajectories)
print("Natural class distribution:")
for cls, count in sorted(class_counts.items()):
    print(f"  {cls:30s}: {count:4d} embryos ({count/len(dataset)*100:.1f}%)")

In [ ]:
# Compare sampling frequency at different gamma values
gammas = [0.0, 0.25, 0.5, 0.75, 1.0]
n_draws = 5000

fig, axes = plt.subplots(1, len(gammas), figsize=(4 * len(gammas), 4), sharey=True)

for ax, gamma in zip(axes, gammas):
    frag_g = FragmentDataset(dataset, min_context=2, horizons=(1,),
                            epoch_length=n_draws, gamma=gamma, n_targets=1)
    
    # Draw samples and count classes
    sampled_classes = []
    for i in range(n_draws):
        s = frag_g[i]
        traj_idx = s["embryo_idx"].item()
        sampled_classes.append(dataset.trajectories[traj_idx].perturbation_class)
    
    sampled_counts = Counter(sampled_classes)
    
    classes = sorted(sampled_counts.keys())
    # Truncate long class names for display
    short_names = [c[:15] for c in classes]
    freqs = [sampled_counts[c] / n_draws * 100 for c in classes]
    natural_freqs = [class_counts[c] / len(dataset) * 100 for c in classes]
    
    x = np.arange(len(classes))
    ax.bar(x - 0.15, natural_freqs, 0.3, label="natural", color="lightgray", edgecolor="k")
    ax.bar(x + 0.15, freqs, 0.3, label="sampled", color="steelblue", edgecolor="k")
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=7)
    ax.set_title(f"γ = {gamma}")
    ax.set_ylabel("Frequency (%)" if gamma == gammas[0] else "")
    if gamma == gammas[0]:
        ax.legend(fontsize=8)

plt.suptitle(f"Class-balanced sampling: natural vs sampled frequency ({n_draws} draws)", fontsize=12)
plt.tight_layout()
plt.show()

### Per-trajectory sampling weights

In [ ]:
# Show how weights are distributed across trajectories
frag_default = FragmentDataset(dataset, min_context=2, horizons=(1,),
                               epoch_length=100, gamma=0.5, n_targets=1)

weights = frag_default._sampling_weights
valid_classes = [dataset.trajectories[i].perturbation_class for i in frag_default.valid_indices]

fig, ax = plt.subplots(figsize=(12, 4))
# Color by class
for cls in sorted(set(valid_classes)):
    mask = [c == cls for c in valid_classes]
    idx = np.where(mask)[0]
    ax.bar(idx, weights[mask], label=cls[:20], alpha=0.7)

ax.axhline(1.0 / len(weights), color="red", ls="--", alpha=0.5, label="uniform")
ax.set_xlabel("Trajectory index (within valid set)")
ax.set_ylabel("Sampling probability")
ax.set_title("Per-trajectory sampling weights (γ=0.5)")
ax.legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

# Print per-class total weight
print("\nPer-class total sampling weight:")
for cls in sorted(set(valid_classes)):
    mask = np.array([c == cls for c in valid_classes])
    total_w = weights[mask].sum()
    n_trajs = mask.sum()
    print(f"  {cls:30s}: weight={total_w:.4f}  (n_trajs={n_trajs}, natural_frac={n_trajs/len(weights):.4f})")

---
## Summary

Key things to check in the plots above:

1. **Trajectory lengths**: Are there enough long trajectories for the sampler to work with?
2. **Inter-frame Δt**: Should be tightly clustered after timestamp repair.
3. **PCA**: How much variance do the first few components capture?
4. **Fragments**: Context windows should be contiguous, targets should lie ahead of context end, and predecessor→target arrows should follow the trajectory.
5. **Class balancing**: At γ=0.5, rare classes should be upweighted (but not to full uniformity).